In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,ConfusionMatrixDisplay, roc_auc_score, roc_curve,precision_recall_curve, average_precision_score)
from xgboost import XGBClassifier
import joblib
import os
os.makedirs('model',exist_ok=True)
from imblearn.over_sampling import SMOTE

df = pd.read_csv(r"C:\Users\niluc\Downloads\PROJECT\Fraud\Data\fraud_clean.csv")
X = df.drop('Class', axis=1)
y =df['Class']
X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Fraud in train: {y_train.sum()}')
print(f'Test: {X_test.shape}   | Fraud in test: {y_test.sum()}')

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_s, y_train)
scale = sum(y_train==0) / sum(y_train==1)
print(f'scale_pos_weight:', scale)
print(f'After SMOTE - Train: {X_train_res.shape}')
print(f'Fraud samples: {y_train_res.sum()} | Legit: {(y_train_res==0).sum()}')
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':        RandomForestClassifier(n_estimators=100, max_depth=10,
                                                   class_weight='balanced',random_state=42, n_jobs=-1),
    'Random Forest+SMOTE' :RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,n_jobs=-1),
    'XGBoost+SMOTE'   :XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,scale_pos_weight=1,
                               sub_sample=0.8,random_state=42,colsample_bytree=0.8,n_jobs=-1,
                               eval_metric='logloss')
}
results={}
for name, m in models.items():
    if 'SMOTE' in name:
        m.fit(X_train_res, y_train_res)
    else:
        m.fit(X_train_s, y_train)
    pred = m.predict(X_test_s)
    prob = m.predict_proba(X_test_s)[:,1]
    auc = roc_auc_score(y_test, prob)
    ap  =average_precision_score(y_test, prob)  #better for imbalanced
    results[name] = {'auc' :auc, 'ap' :ap ,'prob' :prob ,'model' :m, 'pred' :pred}
    print(f'==={name}===')
    print(classification_report(y_test, pred, target_names=['Legit', 'Fraud']))
    print(f'ROC-AUC: {auc:.4f} | Avg Precision: {ap:.4f}')

best_name = max(results, key=lambda k: results[k]['ap'])
best_res  = results[best_name]
best_model = best_res['model']
print(f'\nBest: {best_name} (AP={best_res["ap"]:.4f}, AUC={best_res["auc"]:.4f})')

plt.figure(figsize=(8,5))
for name, res in results.items():
    prec,rec,_ =precision_recall_curve(y_test, res['prob'])
    plt.plot(rec, prec, label=f'{name} (AP={res["ap"]:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='upper right'); 
plt.tight_layout()
plt.savefig('pr_curve.png', dpi=120)

plt.figure(figsize=(7,5))
for name, res in results.items():
    fpr,tpr,_ = roc_curve(y_test, res['prob'])
    plt.plot(fpr, tpr, label=f'{name} ({res["auc"]:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.title('ROC Curves');
plt.legend()
plt.tight_layout();
plt.savefig('fraud_roc_curves.png', dpi=120)


cm = confusion_matrix(y_test, best_model.predict(X_test_s))
ConfusionMatrixDisplay(cm, display_labels=['Legit','Fraud']).plot(cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}')
plt.tight_layout();
plt.savefig('fraud_confusion_matrix.png', dpi=120)

#threshold tuning
print('\nThreshold Analysis:')
for threshold in[0.3, 0.4, 0.5, 0.6]:
    pred_t =(best_res['prob'] >= threshold).astype(int)
    from sklearn.metrics import recall_score, precision_score
    rec = recall_score(y_test, pred_t)
    prec = precision_score(y_test, pred_t)
    print(f'Threshold={threshold}: Recall={rec:.3f} Precision={prec:.3f}')

if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=X.columns)
    fi.nlargest(15).sort_values().plot(kind='barh', figsize=(8,5), color='#1B8CA6')
    plt.title('Top 15 Features')
    plt.tight_layout();
    plt.savefig('fraud_featurea_importances.png', dpi=120)
    print('\nTop 5:\n', fi.nlargest(5).to_string())

    
joblib.dump(best_model,  'fraud_model.pkl')
joblib.dump(scaler,   'fraud_scaler.pkl')
joblib.dump(list(X.columns), 'fraud_feature_names.pkl')
print('Model saved.')


        
        
                                               

Train: (227845, 31) | Fraud in train: 394
Test: (56962, 31)   | Fraud in test: 98
scale_pos_weight: 577.2868020304569
After SMOTE - Train: (454902, 31)
Fraud samples: 227451 | Legit: 227451
===Logistic Regression===
              precision    recall  f1-score   support

       Legit       1.00      0.97      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962

ROC-AUC: 0.9700 | Avg Precision: 0.7109
===Random Forest===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.74      0.83      0.78        98

    accuracy                           1.00     56962
   macro avg       0.87      0.91      0.89     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9733 | Avg Precision: 0.8142
===Random Forest+SMOTE===
    

C:\Users\niluc\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [10:29:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "sub_sample" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


===XGBoost+SMOTE===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.61      0.89      0.72        98

    accuracy                           1.00     56962
   macro avg       0.81      0.94      0.86     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9818 | Avg Precision: 0.8644

Best: XGBoost+SMOTE (AP=0.8644, AUC=0.9818)

Threshold Analysis:
Threshold=0.3: Recall=0.888 Precision=0.442
Threshold=0.4: Recall=0.888 Precision=0.527
Threshold=0.5: Recall=0.888 Precision=0.613
Threshold=0.6: Recall=0.888 Precision=0.685

Top 5:
 V14    0.438709
V10    0.123667
V4     0.061861
V12    0.056117
V8     0.029181
Model saved.
